In [1]:
!pip uninstall -y numpy
!pip install "numpy<2.0"

!pip install torch==2.2.2 torchvision==0.17.2 monai
!pip install scikit-image
!pip install wandb
!pip install import-ipynb
!pip install nibabel 


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/lib/python3.10/dist-packages/numpy-1.26.4.dist-info/' -> '/tmp/pip-uninstall-m6spxmqr'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 105, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", line 675, in uninstall
    uninstalle

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
import monai
import torch
import re
import torchvision.transforms as transforms
from PIL import Image
from monai.transforms import LoadImage
import import_ipynb
from Functions import patients_dicts
from monai.data import MetaTensor

zip_file = "Resources.zip"
os.makedirs("train", exist_ok=True)
!unzip -o -q {zip_file} -d {"train"}
data_path = "train"

full_dict_list = patients_dicts(data_path)

In [4]:
import random

def split_dataset(dataset, per_train=16, per_val=4, seed=None):
    
    if seed is not None:
        random.seed(seed)
        
    catogories_numbers = {}
    catogories_ID = {}

    for patient in dataset:
        disease = patient["Disease"]
        if disease not in catogories_numbers:
            catogories_numbers[disease] = 0
            catogories_ID[disease] = []
        catogories_numbers[disease] += 1
        catogories_ID[disease].append(patient)
        
    split_train_dataset = []
    split_val_dataset = []
    
    for disease, patientIDS in catogories_ID.items():
        random.shuffle(patientIDS)

        train_ids = patientIDS[:per_train]
        val_ids = patientIDS[per_train:per_train+per_val]

        split_train_dataset.extend(train_ids)
        split_val_dataset.extend(val_ids)

    return split_train_dataset, split_val_dataset

In [11]:
from Functions import par_voxelsize, par_size
from monai.transforms import LoadImaged, Compose, EnsureChannelFirstd,ScaleIntensityd, Spacingd,LoadImaged, ResizeWithPadOrCropd

data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel")])

train_dataset_raw = monai.data.Dataset(full_dict_list, transform=data_transform)

voxel_size = []
mean_voxel, std_voxel, max_voxelsize = par_voxelsize(voxel_size,train_dataset_raw)

print("Median voxel size:", mean_voxel)
print("Std voxel size:", std_voxel)
print("Max voxel size:", max_voxelsize)

#sizes = []
#max_size, listofsizes = par_size(sizes,train_dataset_raw)
#print("Max image size:", max_size)
#print(listofsizes)

Median voxel size: [1.5117104 1.5117104 9.335    ]
Std voxel size: [0.18463361 0.18463361 1.6644143 ]
Max voxel size: [ 1.91964  1.91964 10.     ]


In [12]:
from monai.transforms import LoadImaged, Compose, EnsureChannelFirstd,ScaleIntensityd, Spacingd,LoadImaged, ResizeWithPadOrCropd, NormalizeIntensityd

val_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel"),
    Spacingd(keys=["imgED","maskED","imgES","maskES"], pixdim=(mean_voxel[0],mean_voxel[1],mean_voxel[2]), mode=("bilinear", "nearest","bilinear", "nearest"),
                ensure_same_shape=True,align_corners=False),
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True)
])

train_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel"),
    Spacingd(keys=["imgED","maskED","imgES","maskES"], pixdim=(mean_voxel[0],mean_voxel[1],mean_voxel[2]), mode=("bilinear", "nearest","bilinear", "nearest"),
                ensure_same_shape=True,align_corners=False),
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
#    IRENE's MAGIC
])

train_dict_list, val_dict_list = split_dataset(full_dict_list)

val_dataset = monai.data.Dataset(val_dict_list, transform=val_data_transform)
train_dataset = monai.data.Dataset(train_dict_list, transform=train_data_transform)

#for sample in train_dataset:
#    aff = sample["imgED"].meta["affine"]
#    spacing = np.sqrt((aff[:3, :3]**2).sum(0))
#    print( "spacing:", spacing, "shape:", tuple(sample["imgED"].shape))

In [13]:
def visualize_heart_sample(sample, title=None):
    # Visualize the x-ray and overlay the mask, using the dictionary as input
    for i in range(2):
        if i == 0:
            image = np.squeeze(sample['imgED'])
            mask = np.squeeze(sample['maskED'])
        else:
            image = np.squeeze(sample['imgES'])
            mask = np.squeeze(sample['maskES'])

        plt.figure(figsize=[10,7])
        plt.imshow(image, 'gray')

        mask1 = np.ma.masked_where(mask != 1, mask)
        plt.imshow(mask1, 'Greens', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask2 = np.ma.masked_where(mask != 2, mask)
        plt.imshow(mask2, 'Reds', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask3 = np.ma.masked_where(mask != 3, mask)
        plt.imshow(mask3, 'Blues', alpha = 0.5, clim=[0,1], interpolation='nearest')
        if title is not None:
            plt.title(title)
        plt.show()

#shape = train_dataset[0]['imgED'].shape
#for i in range(shape[3]):
#    single_data = train_dataset[0]
#    sample_slice = {
#    'imgED': single_data['imgED'][0,:, :, i],
#    'maskED': single_data['maskED'][0,:, :, i],
#    'imgES': single_data['imgES'][0,:, :, i],
#    'maskES': single_data['maskES'][0,:, :, i]
#    }
#    visualize_heart_sample(sample_slice, title=f"patient:{single_data['ID']},disease:{single_data['Disease']},slice:{i}")